In [0]:

# ==========================================
# LOAD DATASETS
# ==========================================

from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    count,
    coalesce,
    lit
)

# Silver
matches_df = spark.table(
    "workspace.default.silver_matches"
)

# Gold
league_df = spark.table(
    "workspace.default.gold_league_table"
)

performance_df = spark.table(
    "workspace.default.gold_team_performance"
)

summary_df = spark.table(
    "workspace.default.gold_league_summary"
)

# Only completed matches with available scores
finished_df = matches_df.filter(
    (col("match_status") == "FINISHED") &
    col("home_goals").isNotNull() &
    col("away_goals").isNotNull()
)

print("Finished matches:", finished_df.count())
print("League table teams:", league_df.count())
print("Performance teams:", performance_df.count())
print("Summary records:", summary_df.count())


In [0]:
# ==========================================
# MATCH COUNT VALIDATION
# ==========================================

silver_completed = finished_df.count()

gold_completed = summary_df.select(
    "completed_matches"
).first()[0]

print("Silver completed matches:", silver_completed)
print("Gold completed matches:", gold_completed)

assert silver_completed == gold_completed, \
    "FAILED: Match counts do not match!"

print("PASSED: Match counts reconcile!")


In [0]:
# ==========================================
# GOALS RECONCILIATION
# ==========================================

silver_goals = finished_df.select(
    spark_sum(
        col("home_goals") + col("away_goals")
    ).alias("total_goals")
).first()["total_goals"]

gold_goals = summary_df.select(
    "total_goals"
).first()["total_goals"]

print("Silver total goals:", silver_goals)
print("Gold total goals:", gold_goals)

assert silver_goals == gold_goals, \
    "FAILED: Total goals do not reconcile!"

print("PASSED: Total goals reconcile!")


In [0]:
# ==========================================
# TEAM APPEARANCES
# ==========================================

finished_matches = finished_df.count()

performance_totals = performance_df.agg(
    spark_sum("home_played").alias("home_total"),
    spark_sum("away_played").alias("away_total")
).first()

home_total = performance_totals["home_total"]
away_total = performance_totals["away_total"]

print("Finished matches:", finished_matches)
print("Home appearances:", home_total)
print("Away appearances:", away_total)

assert home_total == finished_matches, \
    "FAILED: Home appearances mismatch!"

assert away_total == finished_matches, \
    "FAILED: Away appearances mismatch!"

print("PASSED: Team appearances reconcile!")


In [0]:

# ==========================================
# TEAM POINTS RECONCILIATION
# ==========================================

points_comparison_df = (
    league_df.alias("l")
    .join(
        performance_df.alias("p"),
        ["season_id", "team_id"],
        "inner"
    )
    .select(
        "season_id",
        "team_id",
        col("l.team_name").alias("team_name"),
        col("l.points").alias("league_points"),
        col("p.total_points").alias("calculated_points")
    )
    .withColumn(
        "points_difference",
        col("league_points") - col("calculated_points")
    )
)

display(
    points_comparison_df.orderBy("team_name")
)

# Check how many teams have mismatched points
mismatched_teams = points_comparison_df.filter(
    col("points_difference") != 0
).count()

print("Teams with mismatched points:", mismatched_teams)

assert mismatched_teams == 0, \
    "FAILED: League points do not reconcile!"

print("PASSED: All team points reconcile!")
